In [1]:
%run ./nb_utils_request_api

StatementMeta(, ac383ac8-b978-4e9d-bf2d-45b0a8c0de94, 3, Finished, Available, Finished, True)

In [2]:
%run ./config_api_acto

StatementMeta(, ac383ac8-b978-4e9d-bf2d-45b0a8c0de94, 4, Finished, Available, Finished, True)

In [3]:
import re
import unicodedata

PAYLOAD_PATH = "/lakehouse/default/Files/payloads/payload_osasco_atendimento_cras.json"

ASSUNTO_IGNORAR = {"", "Não se aplica", "Outros"}

DEMANDA_DEPARA = {
    "demanda_servico_de_protecao_social_crascreascentro_pop" : "Proteção Social",
    "demanda_programa_bolsa_familia_pbf"                     : "Bolsa Família",
    "demanda_beneficio_prestacao_continuada_bcp_idoso"       : "BCP Idoso",
    "demanda_beneficio_prestacao_continuada_bcp_pcd"         : "BCP PCD",
    "demanda_carteira_do_idoso"                              : "Cart. Idoso",
    "demanda_carteira_do_autista"                            : "Cart. Autista",
    "demanda_passe_livre_pcd"                                : "Passe Livre",
    "demanda_programa_id_jovem"                              : "ID Jovem",
    "demanda_programa_pe_de_meia"                            : "Pé de Meia",
    "demanda_programa_gas_do_povo"                           : "Gás do Povo",
    "demanda_tarifa_social_de_agua"                          : "Tarifa Água",
    "demanda_programa_bolsa_aluguel"                         : "Bolsa Aluguel",
    "demanda_programa_nosso_futuro_pnf"                      : "Nosso Futuro",
}


def tratar_solicitacoes(df):
    df["data_criacao"] = pd.to_datetime(df["data_criacao"], format="ISO8601")
    df["data_finalizacao"] = pd.to_datetime(df["data_finalizacao"], format="ISO8601")

    df["tempo_atendimento_minutos"] = (
        df["data_finalizacao"] - df["data_criacao"]
    ).dt.total_seconds() / 60

    assunto_cols = [c for c in df.columns if c.startswith("assunto_")]

    def combinar_assuntos(row):
        valores = [
            str(v).strip()
            for v in row[assunto_cols]
            if str(v).strip() not in ASSUNTO_IGNORAR and "Não se aplica" not in str(v)
        ]
        return " + ".join(valores) if valores else None

    df["assunto_combinado"] = df.apply(combinar_assuntos, axis=1)

    demanda_cols = [c for c in DEMANDA_DEPARA if c in df.columns]

    for col in demanda_cols:
        df[col] = df[col].replace("", 0).fillna(0).astype(int)

    def combinar_demandas(row):
        ativas = [DEMANDA_DEPARA[c] for c in demanda_cols if row[c] == 1]
        return " + ".join(sorted(ativas)) if ativas else "Nenhuma"

    df["demanda_combinada"] = df.apply(combinar_demandas, axis=1)

    return df


def tratar_etapas(df):
    colunas_data = [c for c in df.columns if c.startswith("data")]
    for col in colunas_data:
        df[col] = pd.to_datetime(df[col])

    df = df.drop(columns=[
        "codetapa", "tempoexecucao", "tempoexecucaohoras",
        "notifications", "isvalid"
    ])

    df = df.rename(columns={
        "datacriacaoos"    : "data_criacao",
        "datafinalizacaoos": "data_finalizacao",
        "dataetapainicio"  : "data_etapa_inicio",
        "dataetapafim"     : "data_etapa_fim",
        "dataatenderetapa" : "data_atender_etapa",
    })

    duracao = df["data_etapa_fim"] - df["data_etapa_inicio"]
    df["duracao_dias"]     = duracao.dt.days
    df["duracao_horas"]    = duracao.dt.total_seconds() // 3600
    df["duracao_minutos"]  = duracao.dt.total_seconds() // 60
    df["duracao_segundos"] = duracao.dt.total_seconds()

    return df


def salvar_tabela(df, nome_tabela):
    (
        spark.createDataFrame(df)
        .write.mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )


def main():
    df_solicitacoes, df_etapas = extrair_tabela_acto_gestao(PAYLOAD_PATH, TOKEN_OSASCO)

    df_solicitacoes = tratar_solicitacoes(df_solicitacoes)
    df_etapas = tratar_etapas(df_etapas)

    salvar_tabela(df_solicitacoes, "gold_atendimento_cras")
    salvar_tabela(df_etapas, "gold_atendimento_cras_etapas")


main()

StatementMeta(, ac383ac8-b978-4e9d-bf2d-45b0a8c0de94, 5, Finished, Available, Finished, False)

Solicitações: 21
OSs tempo/etapa: 21


In [4]:
display(spark.table("gold_atendimento_cras"))

StatementMeta(, ac383ac8-b978-4e9d-bf2d-45b0a8c0de94, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 49bea1a2-8894-442c-a0b4-a3b574b7647a)